In [1]:
from ace.model import ACE
from pytorch_lightning.utilities.seed import seed_everything
import torch
import matplotlib.pyplot as plt
import scanpy as sc
import anndata
import numpy as np
import pandas as pd
import warnings
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')
import os
from pytorch_lightning.loggers import TensorBoardLogger

os.environ["CUDA_VISIBLE_DEVICES"]="0"

Global seed set to 0
/homes/gws/wqiu0528/anaconda3/envs/ace/lib/python3.9/site-packages/pytorch_lightning/utilities/warnings.py:53: LightningDeprecationWarning: pytorch_lightning.utilities.warnings.rank_zero_deprecation has been deprecated in v1.6 and will be removed in v1.8. Use the equivalent function from the pytorch_lightning.utilities.rank_zero module instead.
  new_rank_zero_deprecation(
/homes/gws/wqiu0528/anaconda3/envs/ace/lib/python3.9/site-packages/pytorch_lightning/utilities/warnings.py:58: LightningDeprecationWarning: The `pytorch_lightning.loggers.base.rank_zero_experiment` is deprecated in v1.7 and will be removed in v1.9. Please use `pytorch_lightning.loggers.logger.rank_zero_experiment` instead.
  return new_rank_zero_deprecation(*args, **kwargs)


In [2]:
adata = anndata.read('../data/ace_mouse_30000cells.h5ad')


In [3]:

adata.obs["age_float"] = adata.obs["age_float"].astype("float")

### train test split
train_size = int(adata.shape[0] * 0.8)
indices = np.arange(adata.shape[0])
np.random.seed(0)
np.random.shuffle(indices)
train_idx = indices[:train_size]
test_idx = indices[train_size:]
train_adata = adata[train_idx, :].copy()
test_adata = adata[test_idx, :].copy()

In [ ]:

sc.pp.pca(test_adata)
for plot_feature in ['age_float', 'cell_type', 'sex', 'tissue']:
    sc.pl.pca(test_adata, color=plot_feature, show=False)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.set_xlabel('')
    ax.set_ylabel('')

In [ ]:
sc.pp.neighbors(test_adata)
sc.tl.umap(test_adata)
for plot_feature in ['age_float', 'cell_type', 'sex', 'tissue']:
    sc.pl.umap(test_adata, color=plot_feature, show=False)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.set_xlabel('')
    ax.set_ylabel('')


In [ ]:

ACE.setup_anndata(train_adata, layer='raw_counts', continuous_phenotype_keys=['age_float'],
categorical_background_keys=['cell_type', 'tissue', 'sex'],
batch_key="batch",
)

model = ACE(
    train_adata,
    n_salient_latent=3,
    n_background_latent=17,
    use_observed_lib_size=True,
    hsic_loss_penalty=1e+6,
    pheno_continuous_recon_penalty=50,
    back_categorical_recon_penalty=[4000, 4000, 12000],
    dropout_rate_encoder=0,
    dropout_rate_pheno=0.5,
    dropout_rate_back=0.5,
)

model.train(
    use_gpu=True,
    check_val_every_n_epoch=1,
    train_size=0.8,
    early_stopping=True,
    early_stopping_monitor="validation_loss",
    early_stopping_patience=45,
    max_epochs=500,
    plan_kwargs=dict(lr=0.0001),
)

In [ ]:
salient_adata = anndata.AnnData(
    X=model.get_latent_representation(test_adata, representation_kind='salient'),
    obs=test_adata.obs
)

In [ ]:
sc.pp.pca(salient_adata)
for plot_feature in ['age_float', 'cell_type', 'sex', 'tissue']:
    sc.pl.pca(salient_adata, color=plot_feature, show=False)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.set_xlabel('')
    ax.set_ylabel('')

In [ ]:

sc.pp.neighbors(salient_adata)
sc.tl.umap(salient_adata)

for plot_feature in ['age_float', 'cell_type', 'sex', 'tissue']:
    sc.pl.umap(salient_adata, color=plot_feature, show=False)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.set_xlabel('')
    ax.set_ylabel('')

In [ ]:
background_adata = anndata.AnnData(
    X=model.get_latent_representation(test_adata, representation_kind='background'),
    obs=test_adata.obs
)
sc.pp.pca(background_adata)
for plot_feature in ['age_float', 'cell_type', 'sex', 'tissue']:
    sc.pl.pca(background_adata, color=plot_feature, show=False)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.set_xlabel('')
    ax.set_ylabel('')

In [ ]:
sc.pp.neighbors(background_adata)
sc.tl.umap(background_adata)

for plot_feature in ['age_float', 'cell_type', 'sex', 'tissue']:
    sc.pl.umap(background_adata, color=plot_feature, show=False)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.set_xlabel('')
    ax.set_ylabel('')
